# TADS23 · Lecture 9 · Deliver: the lumen method as a rule
Step 5 of the information delivery cycle: the graph answers CQ1 and CQ3, and writes the results that Dynamo brings back to Revit (step 6).

The queries are the ones in `queries/cpo-deliver-queries.rq`, read from the repository by their number: L9-1 to L9-7. The same queries run in GraphDB.

**The lumen method:** E = n × Φ × UF × MF / A, with UF = 0.6 and MF = 0.8 as teaching assumptions.

⏸ *Run one cell at a time, when it is said in class.*

## ► Run 1 · Install and read the queries

In [ ]:
# ► Run 1 · Install and read the queries
!pip -q install rdflib
import re, urllib.request
from rdflib import Dataset, URIRef
REPO = 'https://raw.githubusercontent.com/RichZele/cpo/main/'
text = urllib.request.urlopen(REPO + 'queries/cpo-deliver-queries.rq').read().decode('utf-8')
parts = re.split(r'# ═+\n# (L9-\d)[^\n]*\n(?:#[^\n]*\n)*# ═+\n', text)
Q = dict(zip(parts[1::2], parts[2::2]))
print('Queries read:', ', '.join(Q))

## ► Run 2 · The data: four files, each in its own named graph
The ontology, the school (with the catalogue of luminaire types), the Duplex from IFC, and the luminaires placed in the Duplex classrooms.

In [ ]:
# ► Run 2 · The data, as in GraphDB
ds = Dataset(default_union=True)          # like GraphDB: a query without FROM sees all graphs
files = {'graphdb/cpo-all.ttl':           'https://w3id.org/cpo',
         'graphdb/cpo-school-data.ttl':   'https://w3id.org/cpo/data',
         'graphdb/ifcmodel_LBD.ttl':      'https://w3id.org/cpo/duplex',
         'data/cpo-duplex-lighting.ttl':  'https://w3id.org/cpo/duplex-lighting'}
for f, g in files.items():
    ds.graph(URIRef(g)).parse(REPO + f, format='turtle')
    print(f'{len(ds.graph(URIRef(g))):6} triples  {f}')

def show(name):
    rows = ds.query(Q[name])
    print(' | '.join(str(v) for v in rows.vars))
    for r in rows:
        print(' | '.join('' if v is None else str(v) for v in r))

## ► Run 3 · L9-1 · The inputs
For each classroom: the number of luminaires, their total flux, the area and the required illuminance.

In [ ]:
# ► Run 3 · L9-1 · The inputs
show('L9-1')

## ► Run 4 · L9-2 · The rule
A SPARQL INSERT: it calculates E for each classroom and writes the value and a status, PASS or FAIL, into the graph `<https://w3id.org/cpo/results>`.

In [ ]:
# ► Run 4 · L9-2 · The rule
RESULTS = URIRef('https://w3id.org/cpo/results')
ds.update(Q['L9-2'])
print(len(ds.graph(RESULTS)), 'new triples in', RESULTS)
for s, p, o in sorted(ds.graph(RESULTS)):
    print(s.split('/')[-1], p.split('#')[-1].split('/')[-1], o.split('/')[-1] if isinstance(o, URIRef) else o)

## ► Run 5 · L9-3 · CQ1: which classrooms are below the requirement?

In [ ]:
# ► Run 5 · L9-3 · CQ1
show('L9-3')

## ► Run 6 · L9-4 and L9-5 · CQ3: what would fix them?
L9-4: a catalogue type with the same number of luminaires, and the lighting power density (W/m²) before and after. L9-5: more luminaires of the type that is there.

In [ ]:
# ► Run 6 · L9-4 and L9-5 · CQ3
show('L9-4')
print()
show('L9-5')

## ► Run 7 · L9-6 · What Dynamo receives
One row per room of the Revit model, by room number.

In [ ]:
# ► Run 7 · L9-6 · What Dynamo receives
show('L9-6')

## ► Run 8 · The designer swaps the type in A102, and the check runs again
In Revit, the eight luminaires of A102 get the type Multilume Flat Delta 600×600. Here the same change is made in the graph; then L9-7 removes the old results and L9-2 runs again.

In [ ]:
# ► Run 8 · Swap the type in A102, and run the rule again
ds.update('''PREFIX cpo:  <https://w3id.org/cpo#>
PREFIX inst: <https://w3id.org/cpo/data/>
DELETE { GRAPH ?g { ?lum cpo:hasProductType inst:type_23916-529 } }
INSERT { GRAPH ?g { ?lum cpo:hasProductType inst:type_23784-402 } }
WHERE  { GRAPH ?g { ?lum cpo:hasProductType inst:type_23916-529 ;
                         cpo:installedIn <https://w3id.org/cpo/duplex/space_0BTBFw6f90Nfh9rP1dlXr2> } }''')
ds.update(Q['L9-7'])        # remove the old results
ds.update(Q['L9-2'])        # and calculate again
show('L9-6')